<a href="https://colab.research.google.com/github/ilaydacepniogluu-sys/AkademiQ_DataScience/blob/main/AkademiQ4HAFTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import re #metinlerde desen arama ve temizleme
import time #kod çalışma süresini hesaplar.
import warnings #uyarı mesajlarını gizlemek ve yönetmek için
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_predict #çapraz doğrulama ve tahminleme üretmek için
from sklearn.compose import ColumnTransformer #sayısal ve kategorik sütunlara farklı işlemler uygulamak için
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer #eksik verileri doldurmak için
from sklearn.preprocessing import OrdinalEncoder #kategorik verileri sayısala çevirmek için.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor #sayısal değer tahmini yapmak.
from sklearn.ensemble import RandomForestRegressor



In [ ]:
df = pd.read_csv("Food_Delivery_Times.csv")
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


In [12]:
def clean_col(c): #sütun isimlerini temizliyoruz.
  c = c.lower().strip() #lower: tüm harfleri küçültecek. strip:baştaki ve sondaki boşlukları temizleyecek.
  c = re.sub(r"[^a-z0-9]+","_", c) #harf ve rakam dışındaki karakterlerin tamamını _ bununla değiştir.
  return c.strip("_") #baştaki ve sondaki gereksiz alt çizgileri silecek ve tüm sütun adını geri verecek.
df.columns = [clean_col(c) for c in df.columns] # yukarıda yapılan işlemlerin tüm sütunlara uygulanması için bu satırı yazdık.

print(df.columns.tolist())

['order_id', 'distance_km', 'weather', 'traffic_level', 'time_of_day', 'vehicle_type', 'preparation_time_min', 'courier_experience_yrs', 'delivery_time_min']


In [ ]:
#target_candidates =[c for c in df.columns if "time" in c] #target_candidates: hedef sütun seçmek için tanımladığımız yapı. df columns içindeki tüm sütun adlarını tek tek gezsin içinde time geçenleri listeye alsın.

#print(target_candidates)

#TARGET = target_candidates[0] # listede time geçen ilk sütun.


In [ ]:
# Hedef sütunuda temizleme adımından geçiriyoruz.
#TARGET sütununu yeniden temizleyip aynı sütuna geri yazdıracağız.

#df[TARGET]=(
  #  df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float) #.astype(str):sütundaki tüm değerler yazı tipine çevrilecek.
#)

#df =df.dropna(subset=[TARGET]) #target sütununda boş değerler varsa eğer bunları veri setinden sil. dropna(eksik veri sil.)
#print(df[TARGET].head())

In [ ]:
TARGET = "delivery_time_min"
print("Target:",TARGET)

Target: delivery_time_min


In [13]:
df[TARGET]=(
    df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)) #.astype(str):sütundaki tüm değerler yazı tipine çevrilecek.)

In [14]:
df =df.dropna(subset=[TARGET]).reset_index(drop=True) #boş olan target satırlarını sil. (subset=[TARGET]) demek target sütununa bak orda işlem yap demek.
#reset_index : satır numaralarını yeniden düzenleyecek. Karışık görüntü engellenmiş olur.
#drop=true: eski indeks sütun olarak eklenmesin.

print("Target ilk değerler:")
display(df[TARGET].head())

Target ilk değerler:


,delivery_time_min
0,43.0
1,84.0
2,59.0
3,37.0
4,68.0


In [16]:
drop_like_id =[c for c in df.columns if c in {"id","person_id"} or c.endswith("_id")]
df = df.drop(columns=drop_like_id, errors ="ignore") #errors="igonere" demek listedeki sütunlardan biri veri setinde yoksa hata verme direkt geç.


print("ID sonrası şekil:", df.shape)

ID sonrası şekil: (1000, 8)


In [17]:
#Eğitim ve test seti ayırma
# X de girdiler ve inputeler
# y de ise hedefleri veriyoruz.

X = df.drop(columns=[TARGET]).copy() #algoritmaya girdi.
y = df[TARGET].copy() #algoritmanın hedefi .copy ise ana veri setini bozmadan dağıt.



In [18]:
time_col =next((c for c in X.columns if "order" in c and "time" in c), None)

if time_col:
  dt = pd.to_datetime(X[time_col], fotmat="%H:%M:%S", errors ="coerce")
  X["order_hour"] =dt.dt.hour # dt.dt.hour : sadece saat kısmını al. 10:18:24 gibi ifadeden sadece 10 u alacak ve yeni sütun olarak kaydedecek.
  X = X.drop(columns=[time_col]) #eksik verileri kaldır.


  #Eğer yukarıda söylenilen gibi bir sütun bulduysan aşağıdaki kodları çalıştır.
  #errors="coerce": Hatalı bir saat varsa bunu bozma bunu direk olarak boş(NaT) yap.
  #NaT: zamansal verilerde eksiklik anlamına gelir.

In [20]:
lat_cols = [c for c in X.columns if "lat" in c]
lon_cols = [c for c in X.columns if "long" in c or "lng" in c]

if len (lat_cols) >= 2 and len(lon_cols) >= 2:
  r_lat, d_lat = lat_cols[:2] #[restaurant_latitude,delivery_location_latitude]
  r_lon, d_lon = lon_cols[:2]
  X["geo l1 distance"] = (X[r_lat] - X[d_lat]).abs() + (X[r_lon] - X[d_lon]).abs() #.abs() mutlak değer alır.

In [21]:
if "weather" in X.columns:
  X["bad_weather"] = X["weather"].astype(str).str.lower().isin(
      ["rainy","stormy","foggy","snowy"]
  ).astype(int)

In [22]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Kategorik Sütunlar:", cat_cols)
print("Sayısal Sütunlar:", len(num_cols))

Kategorik Sütunlar: ['weather', 'traffic_level', 'time_of_day', 'vehicle_type']
Sayısal Sütunlar: 4


In [23]:
models ={
    "Linear Regsession": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth = 8, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators= 200, max_depth = 8, random_state=42 , n_jobs=-1)
}

